In [42]:
import math
import io
from pathlib import Path
from pypdf import PdfReader, PdfWriter, Transformation, PageObject
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4

def merge_pdfs_content_fixed(root_path, target_file, output_name="cfos_final_fixed.pdf", rows_per_page=6):
    root = Path(root_path)
    pdf_paths = sorted(list(root.glob(f"CFOS*/*/{target_file}")))
    
    if not pdf_paths:
        print("파일을 찾을 수 없습니다.")
        return

    writer = PdfWriter()
    dest_w, dest_h = A4
    grid_h = dest_h / rows_per_page
    
    current_page = None

    for i, path in enumerate(pdf_paths):
        # 1. 페이지 생성 로직
        if i % rows_per_page == 0:
            if current_page:
                writer.add_page(current_page)
            current_page = PageObject.create_blank_page(width=dest_w, height=dest_h)

        reader = PdfReader(path)
        src_page = reader.pages[0]
        
        # 2. 좌표 및 스케일 계산
        src_w = float(src_page.mediabox.width)
        src_h = float(src_page.mediabox.height)
        
        # 라벨 공간 30pt 제외
        scale = min(dest_w / src_w, (grid_h - 30) / src_h) * 0.99
        
        local_idx = i % rows_per_page
        ty_top_of_slot = dest_h - (grid_h * (local_idx + 1))
        
        tx = (dest_w - src_w * scale) / 2
        ty = ty_top_of_slot + (grid_h - 30 - src_h * scale) / 2 +5

        # 3. 새로운 가상 페이지에 원본 병합 (핵심 수정)
        # 원본 페이지를 직접 변형하는 대신, 중간 객체를 거쳐 안전하게 병합합니다.
        temp_page = PageObject.create_blank_page(width=dest_w, height=dest_h)
        op = Transformation().scale(scale).translate(tx, ty)
        temp_page.merge_transformed_page(src_page, op)
        
        # 4. 결과 페이지에 히트맵과 라벨을 순차적으로 병합
        current_page.merge_page(temp_page)
        
        # 라벨 생성 (별도의 캔버스로 정확한 위치에 찍기)
        packet = io.BytesIO()
        can = canvas.Canvas(packet, pagesize=(dest_w, dest_h))
        can.setFont("Helvetica-Bold", 5)
        can.drawCentredString(dest_w / 2, ty_top_of_slot + grid_h - 22, path.parents[1].name)
        can.save()
        packet.seek(0)
        current_page.merge_page(PdfReader(packet).pages[0])

        print(f"Processing: {path.parents[1].name}")

    if current_page:
        writer.add_page(current_page)

    with open(output_name, "wb") as f:
        writer.write(f)
    print(f"\n완료! '{output_name}' 파일을 확인해 보세요.")

# 실행
# merge_pdfs_content_fixed("./results")

# 실행: A4 한 페이지당 5개씩
# merge_pdfs_to_multi_page("./data", rows_per_page=5)
# 실행: 3열 그리드로 배치
# merge_pdfs_to_single_page("./results", cols=3)
# 사용 예시: 한 줄에 2개씩 배치
# merge_pdfs_to_grid("./data", cols=2)
# 실행을 위해 필요한 라이브러리: pip install pypdf reportlab
#_significant_only
target_file = "signal_heatmap_significant_only.pdf"
output_name = "cfos_heatmap_significant_only.pdf"
target_dir = "/home/jhahn/brain-slice-render/web_cfos/projects"
merge_pdfs_content_fixed(target_dir,target_file,target_dir+"/"+output_name)

Processing: CFOSgradient
Processing: CFOSgradient_cor
Processing: CFOSgradient_sag

완료! '/home/jhahn/brain-slice-render/web_cfos/projects/cfos_heatmap_significant_only.pdf' 파일을 확인해 보세요.
